# Predictive Trading Model: XGBoost Regressor with Macro Regime Filtering

This notebook implements an algorithmic trading model using a continuous regressor. It utilizes 50-day Z-Scores to normalize all-time highs and implements a 200-day macro regime filter to identify secular market trends.

The strategy aims to maximize the Sharpe Ratio by timing breakouts and optimizing cash positions during drawdowns.


In [ ]:
!pip install yfinance ta scikit-learn pandas numpy matplotlib seaborn


In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import ta
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


## 1. Download Market & Macro Data


In [ ]:
print("Downloading Data...")
symbols = {
    'RELIANCE.NS': 'Close',
    '^NSEI': 'NIFTY_Close',
    'INR=X': 'INR_Close'
}
dfs = []
for sym, col_name in symbols.items():
    df = yf.download(sym, start='2015-01-01', end='2024-08-07', auto_adjust=False, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.droplevel(1)
    df = df.reset_index()[['Date', 'Close']].rename(columns={'Close': col_name})
    if sym == 'RELIANCE.NS':
        stock = yf.download(sym, start='2015-01-01', end='2024-08-07', auto_adjust=False, progress=False)
        if isinstance(stock.columns, pd.MultiIndex):
            stock.columns = stock.columns.droplevel(1)
        stock = stock.reset_index()
        df = stock[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()
    dfs.append(df)
    
data = dfs[0]
for i in range(1, len(dfs)):
    data = data.merge(dfs[i], on='Date', how='left')

data = data.sort_values('Date').fillna(method='ffill').dropna().reset_index(drop=True)
print(f"Loaded {len(data):,} daily RELIANCE.NS records.")


## 2. Feature Engineering


In [ ]:
data['Return_1d'] = data['Close'].pct_change(1)
data['Return_3d'] = data['Close'].pct_change(3)
data['Return_7d'] = data['Close'].pct_change(7)
data['SMA_14_Ratio'] = data['Close'] / data['Close'].rolling(14).mean()
data['SMA_50_Ratio'] = data['Close'] / data['Close'].rolling(50).mean()
data['Vol_14d'] = data['Return_1d'].rolling(14).std()

# Macro Features
data['NIFTY_Return_1d'] = data['NIFTY_Close'].pct_change(1)
data['NIFTY_Return_7d'] = data['NIFTY_Close'].pct_change(7)
data['NIFTY_SMA_50_Ratio'] = data['NIFTY_Close'] / data['NIFTY_Close'].rolling(50).mean()
data['INR_Return_1d'] = data['INR_Close'].pct_change(1)
data['INR_Return_7d'] = data['INR_Close'].pct_change(7)

# Feature 1: Stationary 50-day Z-Scores (Handles All-Time Highs)
data['Z_Score_50d'] = (data['Close'] - data['Close'].rolling(50).mean()) / data['Close'].rolling(50).std()

# Feature 2: 200-day Macro Regime Filter
data['Regime_200d'] = (data['Close'] > data['Close'].rolling(200).mean()).astype(int)
data['NIFTY_Regime_200d'] = (data['NIFTY_Close'] > data['NIFTY_Close'].rolling(200).mean()).astype(int)

# Auto-generate 80+ TA indicators
data = ta.add_all_ta_features(data, open="Open", high="High", low="Low", close="Close", volume="Volume", fillna=True)

# Feature 3: Continuous Target (Regression)
data['Target'] = data['Close'].pct_change().shift(-1)
data = data.dropna().reset_index(drop=True)

feature_columns = [c for c in data.columns if c not in ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Target', 'NIFTY_Close', 'INR_Close'] and np.issubdtype(data[c].dtype, np.number)]

X = data[feature_columns]
y = data['Target']


## 3. Train-Test Split (80% / 20%)


In [ ]:
split_index = int(len(data) * 0.80)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:-1]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:-1]
test_data = data.iloc[split_index:-1].copy().reset_index(drop=True)


## 4. Train the HistGradientBoostingRegressor


In [ ]:
model = HistGradientBoostingRegressor(max_iter=100, max_depth=3, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Test MSE: {mean_squared_error(y_test, y_pred):.6f}")


## 5. Backtest Strategy & Results


In [ ]:
# If model predicts tomorrow's return > 0, we BUY. Otherwise we SELL/Hold Cash.
test_data['Prediction'] = (y_pred > 0).astype(int)

# Calculate Strategy Returns
test_data['Strategy_Return'] = test_data['Prediction'] * test_data['Return_1d'].shift(-1)
test_data['Position_Change'] = test_data['Prediction'].diff().fillna(0).abs()
test_data['Strategy_Return'] -= test_data['Position_Change'] * 0.001 # 0.1% transaction fee

agent_equity = (1 + test_data['Strategy_Return']).cumprod()
buy_hold_equity = (1 + test_data['Return_1d'].shift(-1)).cumprod()

agent_returns = test_data['Strategy_Return'].dropna().to_numpy()
buy_hold_returns = test_data['Return_1d'].shift(-1).dropna().to_numpy()

agent_sharpe = np.sqrt(365) * agent_returns.mean() / (agent_returns.std() + 1e-9)
buy_hold_sharpe = np.sqrt(365) * buy_hold_returns.mean() / (buy_hold_returns.std() + 1e-9)

initial_capital = 10000
results = pd.DataFrame({
    'Strategy': ['Regressor Model', 'Buy and hold'],
    'Total return': [agent_equity.iloc[-2] - 1, buy_hold_equity.iloc[-2] - 1],
    'Profit (₹)': [(agent_equity.iloc[-2] - 1) * initial_capital, (buy_hold_equity.iloc[-2] - 1) * initial_capital],
    'Annualized Sharpe': [agent_sharpe, buy_hold_sharpe]
})
results['Total return'] = results['Total return'].map('{:.2%}'.format)
results['Profit (₹)'] = results['Profit (₹)'].map('₹{:,.2f}'.format)
results['Annualized Sharpe'] = results['Annualized Sharpe'].map('{:.2f}'.format)

print("Final Performance on RELIANCE.NS (Assuming ₹10,000 starting capital):\n")
from IPython.display import display
display(results)

print("\nAgent Actions (on Test Set):\n")
action_counts = test_data['Prediction'].map({1: 'Buy / Hold RELIANCE', 0: 'Sell / Hold Cash'}).value_counts()
for k, v in action_counts.items():
    print(f"{k}: {v} days")

plt.figure(figsize=(14, 7))
sns.lineplot(x=test_data['Date'].iloc[:-1], y=buy_hold_equity.iloc[:-1], label='Buy and Hold', color='orange')
sns.lineplot(x=test_data['Date'].iloc[:-1], y=agent_equity.iloc[:-1], label='Gradient Boosting Regressor', color='blue')
plt.title('Regressor Model vs Buy & Hold (RELIANCE.NS)')
plt.ylabel('Cumulative Return Multiplier')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
